In [10]:
# Cell 1 - Import libraries
import spacy
import warnings
warnings.filterwarnings('ignore')

# Load english model
nlp = spacy.load('en_core_web_sm')

print("spaCy loaded successfully!")

spaCy loaded successfully!


In [18]:
# Cell 2 - Severity keyword configuration (English-only)
# Simplified lists: remove Tamil transliteration entries per request.
severity_keyword_rules = {
    'no_injury': [
        'no injury', 'no injuries', 'no one injured', 'not injured', 'no bleeding',
        'safe', 'okay', 'conscious', 'alert', 'no casualties'
    ],
    'high': [
        'dead', 'died', 'death', 'unconscious', 'not breathing', "can't breathe",
        'heavy bleeding', 'blood everywhere', 'bleeding badly', 'trapped', 'stuck inside',
        'crushed', 'head injury', 'skull fracture', 'broken neck', 'spine injury',
        'chest injury', 'severe pain', 'multiple fractures', 'body not moving',
        'heart attack', 'seizure', 'fire', 'vehicle burning', 'explosion', 'smoke from car',
        'car upside down', 'rollover', 'fell from bridge', 'fell in ditch', 'hit by lorry',
        'truck crash', 'bus accident', 'high speed crash', 'major accident', 'fatal accident',
        'person under vehicle', 'pinned under bike', 'emergency', 'help immediately',
        'save me', 'critical', 'serious condition', 'help', 'ambulance', 'ambulance needed'
    ],
    'medium': [
        'fracture', 'broken arm', 'broken leg', 'injured', 'hurt', 'wounded', 'broken',
        'pain', 'bleeding', 'deep cut', 'unable to stand', 'unable to walk', 'collision',
        'airbag deployed', 'passenger injured', 'driver injured', 'dizziness', 'fainted',
        'swelling', 'neck pain', 'back pain', 'shoulder pain'
    ],
    'low': [
        'scratch', 'small scratch', 'minor injury', 'slight pain', 'little pain', 'bruises',
        'small cut', 'vehicle damage', 'bumper damage', 'mirror broken', 'indicator broken',
        'slow speed accident', 'parking accident', 'tyre burst', 'skid but safe', 'fell but okay'
    ],
}

print('Severity keyword rules loaded (English-only).')

Severity keyword rules loaded (English-only).


In [19]:
# Cell 3 - Extract accident info from text (PhraseMatcher + negation)
from spacy.matcher import PhraseMatcher
import math

negation_words = set(["no","not","without","never","cant","can't","dont","don't","illa","illai","illaye","illaiy","illaiyya"])


def build_matcher(nlp, rules):
    matcher = PhraseMatcher(nlp.vocab, attr='LOWER')
    for label in ('no_injury','high','medium','low'):
        phrases = rules.get(label, [])
        if phrases:
            patterns = [nlp.make_doc(p) for p in phrases]
            matcher.add(label.upper(), patterns)
    return matcher

matcher = build_matcher(nlp, severity_keyword_rules)


def is_negated(span):
    # small window check for negation tokens or dependency 'neg'
    start = max(0, span.start - 3)
    end = min(len(span.doc), span.end + 3)
    for tok in span.doc[start:end]:
        if tok.lower_ in negation_words or tok.dep_ == 'neg':
            return True
    return False


def extract_accident_info(text):
    doc = nlp(text)
    matches = matcher(doc)
    score = 0
    matched = []

    # if explicit no_injury found, short-circuit to Low
    for match_id, start, end in matches:
        label = nlp.vocab.strings[match_id].lower()
        span = doc[start:end]
        matched.append((label, span.text))
        if label == 'no_injury' or is_negated(span):
            return {
                'severity': 'Low',
                'confidence': 1.0,
                'matched': matched,
                'locations': [ent.text for ent in doc.ents if ent.label_ in ['GPE','LOC','FAC']],
                'people_count': [ent.text for ent in doc.ents if ent.label_ == 'CARDINAL'],
                'original_text': text
            }

    # Score matches
    for label, span_text in matched:
        if label == 'high':
            score += 3
        elif label == 'medium':
            score += 2
        elif label == 'low':
            score -= 1

    # Map score to severity
    if score >= 3:
        severity = 'High'
    elif score >= 2:
        severity = 'Medium'
    else:
        severity = 'Low'

    # confidence: sigmoid over score
    confidence = 1 / (1 + math.exp(-0.8 * (score)))

    return {
        'severity': severity,
        'confidence': round(confidence,3),
        'matched': matched,
        'locations': [ent.text for ent in doc.ents if ent.label_ in ['GPE','LOC','FAC']],
        'people_count': [ent.text for ent in doc.ents if ent.label_ == 'CARDINAL'],
        'original_text': text
    }


# Test it with mixed examples
examples = [
    'No injury, just a small scratch after the crash',
    'Bad accident on Anna Salai near Gemini flyover, 3 people injured, one is unconscious',
    'Two people injured in a bike accident',
    'moochu illa, but vandi eriyudhu, ambulance venum',
    'kai odanjiduchu and thalai odanjiduchu, emergency da'
]
for sample in examples:
    result = extract_accident_info(sample)
    print('Input:', result['original_text'])
    print('Severity:', result['severity'], 'Confidence:', result.get('confidence'))
    print('Matched:', result['matched'])
    print('Locations:', result['locations'])
    print('People count:', result['people_count'])
    print('-')

Input: No injury, just a small scratch after the crash
Severity: Low Confidence: 1.0
Matched: [('no_injury', 'No injury')]
Locations: []
People count: []
-
Input: Bad accident on Anna Salai near Gemini flyover, 3 people injured, one is unconscious
Severity: High Confidence: 0.982
Matched: [('medium', 'injured'), ('high', 'unconscious')]
Locations: ['Gemini']
People count: ['3']
-
Input: Two people injured in a bike accident
Severity: Medium Confidence: 0.832
Matched: [('medium', 'injured')]
Locations: []
People count: ['Two']
-
Input: moochu illa, but vandi eriyudhu, ambulance venum
Severity: High Confidence: 0.917
Matched: [('high', 'ambulance')]
Locations: ['moochu']
People count: []
-
Input: kai odanjiduchu and thalai odanjiduchu, emergency da
Severity: High Confidence: 0.917
Matched: [('high', 'emergency')]
Locations: []
People count: []
-


In [16]:
# Cell 4 - Synthetic evaluation of extractor using current keyword lists
from sklearn.metrics import classification_report, confusion_matrix
import csv
import os

# Build synthetic samples from configured phrases
samples = []
labels = []
for label, phrases in severity_keyword_rules.items():
    for p in phrases:
        # create a few simple variants to simulate short utterances
        samples.append(p)
        labels.append('High' if label == 'high' else ('Medium' if label == 'medium' else 'Low'))
        samples.append(p + ' please')
        labels.append('High' if label == 'high' else ('Medium' if label == 'medium' else 'Low'))

# Run extractor
preds = []
matched_list = []
for t in samples:
    r = extract_accident_info(t)
    preds.append(r['severity'])
    matched_list.append(r.get('matched', []))

# Metrics
print('\nClassification Report:')
print(classification_report(labels, preds, zero_division=0))

cm = confusion_matrix(labels, preds, labels=['High', 'Medium', 'Low'])
print('Confusion matrix (rows=actual High/Medium/Low):')
print(cm)

# Show first 10 mismatches
mismatches = [(t,a,p,m) for t,a,p,m in zip(samples,labels,preds,matched_list) if a!=p]
print(f'\nTotal samples: {len(samples)}, Mismatches: {len(mismatches)}')
for row in mismatches[:10]:
    print('Actual:', row[1], 'Pred:', row[2], 'Text:', row[0], 'Matched:', row[3])

# Save results
os.makedirs('data', exist_ok=True)
out_csv = 'data/synthetic_eval.csv'
with open(out_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['text', 'actual', 'predicted', 'matched'])
    for t,a,p,m in zip(samples, labels, preds, matched_list):
        writer.writerow([t, a, p, str(m)])
print('\nSaved synthetic evaluation to', out_csv)


Classification Report:
              precision    recall  f1-score   support

        High       0.91      0.89      0.90       182
         Low       0.84      1.00      0.91       102
      Medium       1.00      0.79      0.88        76

    accuracy                           0.90       360
   macro avg       0.92      0.89      0.90       360
weighted avg       0.91      0.90      0.90       360

Confusion matrix (rows=actual High/Medium/Low):
[[162   0  20]
 [ 16  60   0]
 [  0   0 102]]

Total samples: 360, Mismatches: 36
Actual: High Pred: Low Text: moochu illa Matched: [('high', 'moochu illa')]
Actual: High Pred: Low Text: moochu illa please Matched: [('high', 'moochu illa')]
Actual: High Pred: Low Text: ezhunthukkave illa Matched: [('high', 'ezhunthukkave illa')]
Actual: High Pred: Low Text: ezhunthukkave illa please Matched: [('high', 'ezhunthukkave illa')]
Actual: High Pred: Low Text: response illa Matched: [('high', 'response illa')]
Actual: High Pred: Low Text: response i

In [ ]:
# Cell 3 - Voice input using Whisper (small model, auto-language)
import whisper
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wav

# Load whisper model (small)
print("Loading Whisper small model (this may take a while)...")
whisper_model = whisper.load_model("small")
print("Whisper small model loaded!")


def record_audio(duration=5, sample_rate=16000):
    print(f"🎤 Recording for {duration} seconds... SPEAK NOW!")
    audio = sd.rec(int(duration * sample_rate), 
                   samplerate=sample_rate, 
                   channels=1, dtype='float32')
    sd.wait()
    print("✅ Recording done!")
    return audio, sample_rate


def transcribe_audio(audio, sample_rate):
    # Save temporarily
    wav.write('temp_audio.wav', sample_rate, audio)
    # Transcribe with auto language detection
    result = whisper_model.transcribe('temp_audio.wav')
    return result['text']

print("Voice system ready! (Whisper small, auto-language)")

Loading Whisper small model (this may take a while)...


100%|███████████████████████████████████████| 461M/461M [01:41<00:00, 4.77MiB/s]


Whisper small model loaded!
Voice system ready! (Whisper small, Tamil forced)


In [6]:
# Cell 4 - Test voice recording and transcription
print("Testing voice input...")
print("You will have 5 seconds to describe an accident")
print("Say something like: 'Accident on Anna Salai, 2 people injured'")
print()

# Record audio
audio, sr = record_audio(duration=5)

# Transcribe
text = transcribe_audio(audio, sr)
print("📝 You said:", text)

# Extract info
result = extract_accident_info(text)
print("\n🚨 Extracted Info:")
print("Severity:", result['severity'])
print("Locations:", result['locations'])
print("People count:", result['people_count'])

Testing voice input...
You will have 5 seconds to describe an accident
Say something like: 'Accident on Anna Salai, 2 people injured'

🎤 Recording for 5 seconds... SPEAK NOW!
✅ Recording done!
📝 You said:  uncertainty again another three people injured one is unconscious

🚨 Extracted Info:
Severity: High
Locations: []
People count: ['three', 'one']
